# BIM v1 placement campaign evaluation

This notebook analyses freshly generated BIM v1 campaign results. The corpus uses weighted QoS with placement, so the compatible lanes are seeded random search and elitist genetic search. Metrics, objectives, penalties, and violations are the gateway's authoritative re-evaluation; immutable execution digests are retained in `runs.csv`.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'experimentation' / 'icsoc').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from experimentation.icsoc import analysis

RESULTS = ROOT / 'experimentation' / 'icsoc' / 'out' / 'results'
FIGURES = RESULTS / 'figures'
runs = analysis.load_results(RESULTS)
ok = analysis.feasible_runs(runs)
print(f'{len(runs)} runs; {len(ok)} authoritative feasible evaluations')

## Reproducibility and termination

Each row pins the BIM instance, IR, engine, profile, protocol, compiler, evaluator, mode, algorithm, effective options, and explicit termination.

In [ ]:
display(analysis.provenance_summary(runs))
rates = analysis.termination_rates(runs)
display(rates)
pivot = rates.pivot(index='engine', columns='termination', values='rate').fillna(0)
ax = pivot.plot.bar(stacked=True, figsize=(8, 4))
ax.set(ylabel='fraction of runs', xlabel='', ylim=(0, 1), title='Explicit BIM v1 termination')
analysis.save_fig(ax.figure, 'termination_rates', FIGURES)

## Authoritative objective quality

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
series = [ok.loc[ok.engine == engine, 'objective_value'].values for engine in analysis.CAMPAIGN_ENGINES]
ax.boxplot(series, tick_labels=[analysis.ENGINE_LABELS[e] for e in analysis.CAMPAIGN_ENGINES], showfliers=False)
ax.set(ylabel='authoritative weighted score (lower is better)', title='Final feasible solutions')
analysis.save_fig(fig, 'objective_boxplots', FIGURES)
display(analysis.improvement_over_baseline(runs).describe())

## Cross-instance comparison and stochastic statistics

In [ ]:
taus, profiles = analysis.performance_profile(runs)
fig, ax = plt.subplots(figsize=(7, 4))
for engine, values in profiles.items():
    ax.plot(taus, values, label=analysis.ENGINE_LABELS[engine], color=analysis.ENGINE_COLORS[engine])
ax.set(xlabel='performance ratio', ylabel='fraction of instances', ylim=(0, 1))
ax.legend()
analysis.save_fig(fig, 'performance_profiles', FIGURES)

ranks, paired_test = analysis.mean_ranks(runs)
display(ranks.to_frame())
print(paired_test)
display(analysis.pairwise_engine_stats(runs))